# Guía de apoyo — Práctico Power BI Desktop (E2B / AdventureWorksDW)

> Nombre: Juan Flores

---

## PARTE A — Entorno y exploración del modelo

**A1. Tablas de hechos vs. dimensiones**
- **Tablas de hechos:** `FactInternetSales`, `FactCurrencyRate`
- **Tablas de dimensión:** `DimCustomer`, `DimProduct`, `DimProductCategory`, `DimProductSubcategory`, `DimDate`, `DimEmployee`, `DimCurrency`
- **Criterio técnico:** las tablas de hechos contienen las claves foráneas (FK) que apuntan a las dimensiones, columnas numéricas cuantitativas (montos, cantidades, costos) y tienen muchas más filas. Las dimensiones tienen una clave primaria única, atributos descriptivos (texto) y pocas filas en comparación.

**A2. Motor de almacenamiento**
Power BI usa el motor **VertiPaq** (motor tabular en memoria, columnar). Dos características clave:
1. **Almacenamiento columnar con compresión** (codificación por diccionario, RLE): agrupa valores por columna en vez de por fila, lo que reduce drásticamente el tamaño en disco/memoria.
2. **Procesamiento en memoria (in-memory)**: al mantener todo el modelo comprimido en RAM, los cálculos de agregación sobre millones de filas se resuelven en milisegundos, sin leer disco.

**A3. Importar DimGeography desde SQL Server**
1. Cinta **Inicio (Home) → Obtener datos (Get Data) → SQL Server**.
2. En el cuadro de diálogo, ingresar **Servidor** y **Base de datos** (opcional: usuario/clave o autenticación de Windows).
3. Elegir el modo de conectividad **Importar (Import)**, no DirectQuery.
4. En la ventana **Navegador (Navigator)**, marcar la tabla `DimGeography` y hacer clic en **Cargar** (o **Transformar datos** si se quiere editar antes).

---

## PARTE B — Transformación con Power Query

**B1. Combinar FirstName + LastName**
1. Editor de Power Query → consulta `DimCustomer`.
2. Seleccionar las columnas `FirstName` y `LastName` (Ctrl+clic).
3. Clic derecho → **Combinar columnas (Merge Columns)** → separador: **Espacio** → nombre nueva columna: `NombreCompleto`.
4. Esto queda registrado automáticamente como un paso en **Pasos aplicados (Applied Steps)**.

**B2. DimProduct**
- (a) Nulos en `Color`: seleccionar la columna → pestaña **Transformar → Reemplazar valores**. En "Valor a buscar" escribir `null` (Power Query lo interpreta como el valor nulo real) y en "Reemplazar por" escribir `NA`.
- (b) `ListPrice`: verificar el ícono de tipo de dato en el encabezado; si no es **Número decimal fijo (moneda)**, clic derecho → **Cambiar tipo → Número decimal fijo**.

**B3. Consulta por referencia VentasPorProducto**
1. Clic derecho en `FactInternetSales` → **Referencia (Reference)**.
2. Renombrar la nueva consulta a `VentasPorProducto`.
3. Seleccionar la columna `ProductKey` → **Transformar → Agrupar por (Group By)**.
4. Usar **Avanzado (Advanced)** para agregar dos agregaciones:
   - Nombre nueva columna: `TotalVentas`, Operación: **Suma**, Columna: `SalesAmount`.
   - Nombre nueva columna: `NumTransacciones`, Operación: **Contar filas (Count Rows)**.

---

## PARTE C — Modelado de datos

**C1. Relaciones (vista de Modelo)**
Arrastra la columna clave de la tabla de hechos hacia la clave correspondiente en la dimensión (o usa **Administrar relaciones**). Verifica en cada una: cardinalidad **Varios a uno (*:1)** y dirección de filtro **Única**:
- `FactInternetSales[ProductKey] → DimProduct[ProductKey]`
- `FactInternetSales[CustomerKey] → DimCustomer[CustomerKey]`
- `FactInternetSales[OrderDateKey] → DimDate[DateKey]` (debe quedar **activa**, línea continua)
- `FactInternetSales[CurrencyKey] → DimCurrency[CurrencyKey]`
- `FactCurrencyRate[DateKey] → DimDate[DateKey]` y `FactCurrencyRate[CurrencyKey] → DimCurrency[CurrencyKey]`
- `DimProduct[ProductSubcategoryKey] → DimProductSubcategory[ProductSubcategoryKey]` y `DimProductSubcategory[ProductCategoryKey] → DimProductCategory[ProductCategoryKey]`

**C2. ¿Es estrella pura?**
No. La cadena `DimProductCategory → DimProductSubcategory → DimProduct` es un **esquema copo de nieve (snowflake)**, porque la dimensión de producto está normalizada en tres tablas relacionadas en vez de una sola tabla plana. El libro (cap. 4) propone **desnormalizar usando Power Query**: hacer *merge* de las columnas de nombre de categoría y subcategoría directamente dentro de la consulta `DimProduct`, eliminando así las tablas intermedias y dejando una sola dimensión plana (estrella pura).

**C3. Modelo amigable**
- (a) Seleccionar `DimDate` → pestaña **Herramientas de tabla → Marcar como tabla de fechas** → columna `FullDateAlternateKey`.
- (b) En cada columna clave (ProductKey, CustomerKey, DateKey, CurrencyKey, OrderDateKey, DueDateKey, ShipDateKey, ProductSubcategoryKey, ProductCategoryKey): clic derecho → **Ocultar en vista de informe**.
- (c) Seleccionar `EnglishMonthName` → **Herramientas de columna → Ordenar por columna → MonthNumberOfYear**.
- (d) En `DimDate`, arrastrar `CalendarYear` sobre sí misma → **Crear jerarquía**, renombrar a `Calendario`, luego arrastrar `CalendarQuarter` y `EnglishMonthName` dentro, en ese orden.

---

## PARTE D — Columnas calculadas con DAX

**D1.**
```DAX
Subcategoria = RELATED(DimProductSubcategory[EnglishProductSubcategoryName])
```
Se usa `RELATED` porque `DimProduct` está del lado "muchos" de la relación con `DimProductSubcategory` (lado "uno"): cada fila de producto tiene exactamente una subcategoría relacionada, por lo que `RELATED` puede navegar sin ambigüedad hacia el lado "uno".

**D2.**
```DAX
RangoPrecio =
IF ( DimProduct[ListPrice] > 1000, "Alto",
    IF ( DimProduct[ListPrice] > 100, "Medio", "Bajo" )
)
```
Los precios en blanco quedan en "Bajo" automáticamente, porque `BLANK() > 1000` y `BLANK() > 100` se evalúan como `FALSE`.

**D3.**
```DAX
AnioNacimiento = YEAR ( DimCustomer[BirthDate] )
EmailMayus = UPPER ( DimCustomer[EmailAddress] )
```
`YEAR` pertenece a las funciones de **Fecha y hora**; `UPPER` pertenece a las funciones de **Texto** (cap. 5).

---

## PARTE E — Medidas con DAX

**E1.**
```DAX
Total Ventas = SUM ( FactInternetSales[SalesAmount] )
Total Costo = SUM ( FactInternetSales[TotalProductCost] )
```
Formato: moneda en ambas (panel de propiedades → Formato → Moneda).

**E2.**
```DAX
Margen % =
VAR Ventas = [Total Ventas]
VAR Costo = [Total Costo]
RETURN
DIVIDE ( Ventas - Costo, Ventas )
```
Formato: porcentaje, 2 decimales.

**E3.**
```DAX
Ticket Promedio =
DIVIDE ( [Total Ventas], DISTINCTCOUNT ( FactInternetSales[SalesOrderNumber] ) )
```

**E4.**
```DAX
% Participación Ventas =
DIVIDE (
    [Total Ventas],
    CALCULATE ( [Total Ventas], ALL ( DimProduct ), ALL ( DimProductSubcategory ), ALL ( DimProductCategory ) )
)
```
Se eliminan los filtros de las tres tablas de producto para que el denominador sea siempre el total general, sin importar qué categoría/subcategoría/producto esté filtrado. Verifícala en una matriz con `EnglishProductCategoryName` en filas: la suma de los porcentajes de todas las categorías debe dar 100%.

**E5. Contexto de fila vs. contexto de filtro (conceptual)**
- **Contexto de fila:** existe cuando DAX evalúa una expresión fila por fila (como en una columna calculada o dentro de un iterador). No hay agregación implícita; la fórmula "sabe" en qué fila está.
- **Contexto de filtro:** es el conjunto de filtros (de la matriz, segmentaciones, filtros de página/visual) que determina qué filas se agregan al calcular una medida.
- La medida **E1** (`Total Ventas`) usa **contexto de filtro**: `SUM` agrega las filas visibles según los filtros activos en cada celda del visual.
- La columna calculada **D2** (`RangoPrecio`) usa **contexto de fila**: se evalúa una vez por cada producto, sin depender de ningún filtro externo.

---

## PARTE F — Inteligencia de tiempo

**F1.**
```DAX
Ventas YTD = TOTALYTD ( [Total Ventas], DimDate[FullDateAlternateKey] )
```

**F2.**
```DAX
Ventas Año Anterior =
CALCULATE ( [Total Ventas], SAMEPERIODLASTYEAR ( DimDate[FullDateAlternateKey] ) )
```
(equivalente con `DATEADD(DimDate[FullDateAlternateKey], -1, YEAR)`)

Diferencia DATEADD vs. PARALLELPERIOD cuando el período es parcial: **DATEADD** desplaza exactamente el mismo rango de días seleccionado (si el período actual es parcial, ej. solo enero, el resultado también será solo enero del año anterior). **PARALLELPERIOD** siempre devuelve el **período completo** (el mes/trimestre/año entero), aunque la selección actual sea parcial.

**F3. Tasa Cierre (patrón LASTNONBLANK, semiaditiva)**
```DAX
Tasa Cierre =
CALCULATE (
    SUM ( FactCurrencyRate[EndOfDayRate] ),
    LASTNONBLANK ( DimDate[DateKey], CALCULATE ( SUM ( FactCurrencyRate[EndOfDayRate] ) ) )
)
```
Esto devuelve, para cualquier período seleccionado, el valor de la última fecha con datos dentro de ese período (no la suma de todas las fechas).

---

## PARTE G — Dashboard y visualizaciones

Renombra la página a **Ventas** (doble clic en la pestaña de la página).

- **G1(a) Matriz:** jerarquía `Calendario` en filas; valores: `Total Ventas`, `Ventas YTD`, `Margen %`. Los íconos de expandir/contraer (drill) se habilitan automáticamente al usar una jerarquía; verifica que el ícono de "doble flecha" (Ir al siguiente nivel) esté visible en el encabezado del visual.
- **G1(b) Columnas agrupadas:** eje X = `EnglishProductCategoryName`, eje Y = `Total Ventas`, leyenda = `CalendarYear`. Panel **Análisis (Analytics)** → **Línea de promedio (Average line)** → Agregar.
- **G1(c) Líneas:** eje X = `EnglishMonthName`, eje Y = `Total Ventas` y `Ventas Año Anterior`.

**G2. Interactividad**
- Agrega dos segmentaciones (Slicer): una con `CalendarYear`, otra con `EnglishProductCategoryName`.
- Selecciona el slicer de categoría → pestaña **Formato → Editar interacciones**. Sobre la matriz deja el ícono de **Filtrar**; sobre el gráfico de columnas selecciona el ícono **Ninguno**.
- **Filtrar** vs. **Resaltar**: *Filtrar* oculta del visual los datos que no cumplen la selección (recalcula el visual solo con esos datos); *Resaltar* mantiene todas las categorías visibles pero resalta en color más oscuro/intenso la porción correspondiente a la selección, dejando el resto atenuado como referencia visual.

**G3. Panel de Filtros**
- (a) Filtro de **página**: agrega `CalendarYear` al panel de filtros de página → tipo **Top N** = 2, según `Total Ventas` (o filtro básico marcando los dos años más recientes con datos).
- (b) Filtro de **visual** sobre la matriz: agrega `Total Ventas` al pozo de filtros del visual → tipo básico → **no está en blanco**.
- (c) Los tres ámbitos del panel de Filtros son: **Filtros de nivel visual** (afectan solo al visual seleccionado), **Filtros de nivel de página** (afectan a todos los visuales de la página activa) y **Filtros de nivel de informe** (afectan a todas las páginas del informe).

---

## PARTE H — Publicación y actualización (conceptual)

**H1.**
- (a) Seleccionar `DimDate[CalendarYear]` → **Herramientas de columna → Resumir por → No resumir**.
- (b) Propiedad **Categoría de datos (Data Category)** (en Herramientas de columna) configurada como `City` o `Country` según corresponda, para que el mapa de Bing geolocalice correctamente los valores.

**H2.**
- (a) **Inicio → Publicar**, elegir el área de trabajo destino. Al publicarse aparecen en el área de trabajo: el **informe (report)** y el **conjunto de datos/modelo semántico (dataset)** asociados.
- (b) Dos formas de agregar mosaicos a un dashboard: **anclar (pin)** una visualización existente desde un informe, o **agregar un mosaico manualmente** (contenido web, imagen, video, dato de streaming, etc.) desde el propio dashboard.
- (c) Al hacer clic sobre un mosaico anclado, se navega al informe/visual original que lo generó (o al contenido enlazado, si es un mosaico personalizado).

**H3.**
- (a) **Compartir un dashboard** da acceso de solo lectura a un dashboard puntual a personas específicas; las **áreas de trabajo de grupo (workspaces)** son espacios colaborativos donde varios usuarios editan y administran el contenido en conjunto. Ambas requieren licencia **Power BI Pro** (o capacidad Premium) para los miembros que colaboran/comparten fuera del plan gratuito individual.
- (b) Se debe instalar el **Gateway de datos local (On-premises data gateway)** para programar la actualización desde una base de datos local.
- (c) Un origen alojado en OneDrive se sincroniza/actualiza automáticamente de forma aproximada **cada hora**.

---

## PARTE I — Análisis e interpretación

Guía para redactar (usa tus propios visuales y cifras reales del modelo):
- **I1:** identifica en el gráfico de columnas (o matriz) qué categoría tiene la barra/valor más alto de `Total Ventas`, y lee su `% Participación Ventas` correspondiente.
- **I2:** observa el gráfico de líneas mensual: busca picos (posible estacionalidad de fin de año, por ejemplo) y valles; compara si el mismo mes se repite alto/bajo entre distintos años usando la leyenda de años.
- **I3:** compara el último año completo con datos vs. el anterior usando `Total Ventas` y `Ventas Año Anterior`; calcula la variación porcentual aproximada `(Actual - Anterior) / Anterior` y redacta una recomendación de negocio (ej. reforzar categorías en crecimiento, promociones en meses bajos, etc.).

---

## PREGUNTA EXTRA

1. Crea la relación `FactInternetSales[ShipDateKey] → DimDate[DateKey]`: quedará **inactiva** automáticamente (línea punteada), porque `DimDate` ya tiene una relación activa por `OrderDateKey`.
2. Medida:
```DAX
Ventas por Fecha Envío =
CALCULATE (
    [Total Ventas],
    USERELATIONSHIP ( FactInternetSales[ShipDateKey], DimDate[DateKey] )
)
```
`USERELATIONSHIP` es la función que activa temporalmente una relación inactiva, solo dentro del contexto de esa fórmula.

---

### Recordatorios de la rúbrica (evita penalizaciones)
- No inviertas relaciones ni uses cardinalidad/dirección incorrecta (−2 pts).
- No calcules como columna algo que debe ser medida, ni viceversa (−1 pt c/u).
- Usa exactamente los nombres pedidos para medidas, columnas, jerarquías y páginas (−0.5 pts c/u si difieren).
- Mantén el dashboard ordenado y con sentido analítico (−2 pts si se ve saturado).